# 🚔 Multi-Camera Incident Investigation Console (Live Interactive Demo)

This notebook launches the complete end-to-end multi-camera surveillance system:
- **1 Main Video**: Incident scene (detects vehicles, runs kinematic incident detection).
- **Multiple Background CCTV Videos**: Surrounding road network.
- **Cross-Camera Journey Retrieval**: Reconstructs the suspect vehicle's path across cameras backward in time using FastReID embeddings and spatio-temporal priors.

---
### Step 1: Clone Repository & Check GPU

In [ ]:
%cd /content
!if [ -d "Vehicle-Tracking" ]; then cd Vehicle-Tracking && git pull origin main; else git clone https://github.com/neoaltair/Vehicle-Tracking.git; fi
%cd /content/Vehicle-Tracking

import torch

print("✅ CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Running on CPU. For best performance, go to Runtime -> Change runtime type -> T4 GPU.")

### Step 2: Install Dependencies & FastReID

In [ ]:
!pip install -q -e ".[dev]"
!pip install -q fastreid==1.4.0 --no-deps
# Download cloudflared for reliable web tunneling (never drops JavaScript chunks)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("✅ Dependencies and Cloudflare Tunnel installed.")

### Step 3: Download FastReID Checkpoint (One-Time, ~198 MB)

In [ ]:
from pathlib import Path

weights_dir = Path("outputs/weights")
weights_dir.mkdir(parents=True, exist_ok=True)
weights_file = weights_dir / "veri_sbs_R50-ibn.pth"

if not weights_file.exists():
    print("Downloading VeRi-776 FastReID weights...")
    !wget -q -O outputs/weights/veri_sbs_R50-ibn.pth https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth
    print("✅ Downloaded weights.")
else:
    print("✅ Found existing weights:", weights_file)

### Step 4: Launch Web Console

Run this cell and click the **`https://...trycloudflare.com`** link printed below. It opens immediately with no passwords and no JavaScript errors!

In [ ]:
import re
import subprocess

# 1. Start Streamlit server in background
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

# 2. Start Cloudflare Tunnel (100% reliable, no dropped JS modules)
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stderr=subprocess.PIPE,
    text=True
)

# 3. Extract and display the public link
for line in tunnel_proc.stderr:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print("\n" + "=" * 65)
        print("🚀 MULTI-CAMERA INVESTIGATION CONSOLE READY:")
        print(f"👉 CLICK TO OPEN: {url}")
        print("=" * 65 + "\n")
        break

# Keep alive
tunnel_proc.wait()